# Modelado — Predicción de Malnutrición a 12 meses EC
## PMCI / Fundación Canguro

**Estrategia**: Cascada temporal con LightGBM
- **F0** → solo prenatal/parto
- **F1** → + nacimiento
- **F2** → + hospitalización
- **F3** → + 40 semanas
- **F4** → + 3 meses
- **F5** → + 6 meses
- **F6** → + 9 meses

**Outcomes**: Stunting (HAZ<-2) | Bajo peso (WAZ<-2) | Wasting (WHZ<-2)

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import shap
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, precision_recall_curve,
    average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

PATH = '/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/KMC-70k-93-2024-Malnutricion-conVel-DATA-SPSS-20250322.xlsx'
OUT  = '/Users/herjimenez/Documents/MAESTRIA/PROYECTOS/Trabajo de grado/'
PLAN = OUT + 'feature_plan.json'

print('Cargando datos...')
df_raw = pd.read_excel(PATH)
df = df_raw.replace('#NULL!', np.nan).copy()
for col in df.columns:
    c = pd.to_numeric(df[col], errors='coerce')
    if df[col].notna().sum() == 0 or c.notna().sum() / df[col].notna().sum() >= 0.5:
        df[col] = c

with open(PLAN) as f:
    plan = json.load(f)

FASES = plan['fases']
print(f'Dataset: {df.shape[0]:,} x {df.shape[1]:,}')
print(f'Fases cargadas: {list(FASES.keys())}')

Cargando datos...


Dataset: 64,801 x 753
Fases cargadas: ['F0_Prenatal_Parto', 'F1_Nacimiento', 'F2_Hospitalizacion', 'F3_40semanas', 'F4_3meses', 'F5_6meses', 'F6_9meses']


## 1. Preprocesamiento y Variables Objetivo

In [2]:
# Variables objetivo binarias
df['stunting12m']      = np.where(df['zscoretalla12cat'].notna(),
                                   (df['zscoretalla12cat'] == 1.0).astype(float), np.nan)
df['underweight12m_b'] = np.where(df['zscorepeso12cat'].notna(),
                                   (df['zscorepeso12cat'] == 1.0).astype(float), np.nan)
df['wasting12m']       = np.where(df['zscorepesotalla12cat'].notna(),
                                   (df['zscorepesotalla12cat'] == 1.0).astype(float), np.nan)

OUTCOMES = {
    'Stunting':    ('stunting12m',      '#e74c3c', 3.1),
    'Bajo_peso':   ('underweight12m_b', '#e67e22', 8.2),
    'Wasting':     ('wasting12m',        '#f39c12', 23.2),
}

# Variables leakage (derivadas de datos de 12m o posteriores)
LEAKAGE = {
    'velocidad12_9mesesOMS',   # velocidad 9→12m (requiere dato de 12m)
    'vino12m', 'Desercionreal12meses',
    'rehosp40a12meses', 'mortalidad40sem12meses',
    'indexnutricion12meses', 'MUERTE1ANO',
    'examenneurodurante12meses', 'examenneuropsico12meses',
    'riesgoPC12m',
}
# Todas las variables con '12' en el nombre son del futuro
LEAKAGE |= {c for c in df.columns if '12' in str(c) and c not in
            ['stunting12m','underweight12m_b','wasting12m']}

# Construir feature sets acumulados por fase (cascada)
FASE_ORDER = ['F0_Prenatal_Parto','F1_Nacimiento','F2_Hospitalizacion',
              'F3_40semanas','F4_3meses','F5_6meses','F6_9meses']

cumulative_features = {}
acum = []
for fase in FASE_ORDER:
    cols = FASES.get(fase, [])
    nuevas = [c for c in cols if c in df.columns and c not in LEAKAGE]
    acum = acum + [c for c in nuevas if c not in acum]
    cumulative_features[fase] = list(acum)

print('Feature sets acumulados por fase:')
for fase, cols in cumulative_features.items():
    print(f'  {fase}: {len(cols)} features')

Feature sets acumulados por fase:
  F0_Prenatal_Parto: 41 features
  F1_Nacimiento: 75 features
  F2_Hospitalizacion: 107 features
  F3_40semanas: 137 features
  F4_3meses: 164 features
  F5_6meses: 183 features
  F6_9meses: 198 features


## 2. Funciones de Entrenamiento y Evaluación

In [3]:
def get_model_data(df, features, target_col, min_samples=100):
    """Retorna X, y con solo filas que tienen el outcome."""
    cols = [c for c in features if c in df.columns]
    sub  = df[cols + [target_col]].dropna(subset=[target_col]).copy()
    X    = sub[cols]
    y    = sub[target_col].astype(int)
    return X, y


def train_lgbm_cv(X, y, n_splits=5, spw=None, seed=42):
    """
    Entrena LightGBM con validacion cruzada estratificada.
    Retorna metricas por fold y predicciones OOF (Out-Of-Fold).
    """
    n_pos = y.sum()
    n_neg = len(y) - n_pos
    scale_pos_weight = spw if spw else round(n_neg / n_pos, 2)

    params = {
        'objective':         'binary',
        'metric':            'auc',
        'learning_rate':     0.05,
        'num_leaves':        63,
        'max_depth':         -1,
        'min_child_samples': 30,
        'feature_fraction':  0.8,
        'bagging_fraction':  0.8,
        'bagging_freq':      5,
        'reg_alpha':         0.1,
        'reg_lambda':        0.1,
        'scale_pos_weight':  scale_pos_weight,
        'verbose':           -1,
        'seed':              seed,
    }

    skf     = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    metrics = []
    oof_pred = np.zeros(len(y))
    models  = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)

        model = lgb.train(
            params, dtrain,
            num_boost_round=500,
            valid_sets=[dval],
            callbacks=[
                lgb.early_stopping(50, verbose=False),
                lgb.log_evaluation(-1),
            ]
        )
        models.append(model)

        prob = model.predict(X_val)
        oof_pred[val_idx] = prob

        auc = roc_auc_score(y_val, prob)
        thr = 0.5
        pred_bin = (prob >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_val, pred_bin).ravel()
        metrics.append({
            'fold':        fold + 1,
            'AUC':         round(auc, 4),
            'Sens':        round(tp / (tp+fn) if tp+fn > 0 else 0, 4),
            'Spec':        round(tn / (tn+fp) if tn+fp > 0 else 0, 4),
            'F1':          round(f1_score(y_val, pred_bin, zero_division=0), 4),
            'Precision':   round(precision_score(y_val, pred_bin, zero_division=0), 4),
            'n_train':     len(y_tr),
            'n_val':       len(y_val),
            'best_iter':   model.best_iteration,
        })

    return pd.DataFrame(metrics), oof_pred, models


print('Funciones definidas: get_model_data, train_lgbm_cv')

Funciones definidas: get_model_data, train_lgbm_cv


## 3. Cascada Temporal — Todos los Outcomes

In [4]:
# Entrenar LightGBM para cada fase × cada outcome
# Almacena resultados, predicciones OOF y mejores modelos

resultados  = {}   # (outcome, fase) -> DataFrame de metricas por fold
oof_preds   = {}   # (outcome, fase) -> array de probabilidades OOF
best_models = {}   # (outcome, fase) -> lista de modelos (folds)
data_store  = {}   # (outcome, fase) -> (X, y)

for outcome_name, (target_col, color, _) in OUTCOMES.items():
    print(f'\n========== {outcome_name} ==========')
    for fase in FASE_ORDER:
        feats = cumulative_features[fase]
        X, y  = get_model_data(df, feats, target_col)

        if len(y) < 200 or y.sum() < 20:
            print(f'  {fase}: insuficientes datos (n={len(y)}, pos={y.sum()}) — omitida')
            continue

        metrics_df, oof, models = train_lgbm_cv(X, y)
        auc_mean = metrics_df['AUC'].mean()
        auc_std  = metrics_df['AUC'].std()

        resultados [(outcome_name, fase)] = metrics_df
        oof_preds  [(outcome_name, fase)] = (oof, y)
        best_models[(outcome_name, fase)] = models
        data_store [(outcome_name, fase)] = (X, y)

        sens_m = metrics_df['Sens'].mean()
        spec_m = metrics_df['Spec'].mean()
        print(f'  {fase:<22}: AUC={auc_mean:.4f}±{auc_std:.4f}  '
              f'Sens={sens_m:.3f}  Spec={spec_m:.3f}  n={len(y):,}')


========== Stunting ==========


  F0_Prenatal_Parto     : AUC=0.6454±0.0102  Sens=0.509  Spec=0.689  n=30,953


  F1_Nacimiento         : AUC=0.7374±0.0056  Sens=0.641  Spec=0.708  n=30,953


  F2_Hospitalizacion    : AUC=0.7405±0.0063  Sens=0.639  Spec=0.713  n=30,953


  F3_40semanas          : AUC=0.7678±0.0081  Sens=0.621  Spec=0.758  n=30,953


...

  F2_Hospitalizacion    : AUC=0.7054±0.0142  Sens=0.326  Spec=0.895  n=29,828


  F3_40semanas          : AUC=0.7253±0.0079  Sens=0.331  Spec=0.904  n=29,828


  F4_3meses             : AUC=0.8260±0.0124  Sens=0.500  Spec=0.905  n=29,828


  F5_6meses             : AUC=0.8950±0.0101  Sens=0.601  Spec=0.935  n=29,828


  F6_9meses             : AUC=0.9245±0.0100  Sens=0.683  Spec=0.949  n=29,828


## 4. Comparación de AUC por Fase y Outcome

In [5]:
# Tabla resumen de AUC media por fase y outcome
rows = []
for (outcome, fase), mdf in resultados.items():
    rows.append({
        'Outcome': outcome,
        'Fase':    fase,
        'AUC_mean':  mdf['AUC'].mean(),
        'AUC_std':   mdf['AUC'].std(),
        'Sens_mean': mdf['Sens'].mean(),
        'Spec_mean': mdf['Spec'].mean(),
        'F1_mean':   mdf['F1'].mean(),
        'N_total':   resultados[(outcome,fase)]['n_train'].iloc[0] +
                     resultados[(outcome,fase)]['n_val'].iloc[0],
    })

summary = pd.DataFrame(rows)
pivot_auc = summary.pivot(index='Fase', columns='Outcome', values='AUC_mean')
# Reordenar filas por fase
fase_order_present = [f for f in FASE_ORDER if f in pivot_auc.index]
pivot_auc = pivot_auc.loc[fase_order_present]

print('AUC media por fase y outcome (5-fold CV):')
print(pivot_auc.round(4).to_string())

# Tabla completa de metricas
print('\nTabla completa de metricas:')
tbl = summary[['Outcome','Fase','AUC_mean','AUC_std','Sens_mean','Spec_mean','F1_mean']]
tbl = tbl.round(4)
print(tbl.to_string(index=False))

AUC media por fase y outcome (5-fold CV):
Outcome             Bajo_peso  Stunting  Wasting
Fase                                            
F0_Prenatal_Parto      0.6183    0.6454   0.5550
F1_Nacimiento          0.7509    0.7374   0.6887
F2_Hospitalizacion     0.7538    0.7405   0.7054
F3_40semanas           0.7725    0.7678   0.7253
F4_3meses              0.8737    0.8209   0.8260
F5_6meses              0.9360    0.8935   0.8950
F6_9meses              0.9634    0.9290   0.9245

Tabla completa de metricas:
  Outcome               Fase  AUC_mean  AUC_std  Sens_mean  Spec_mean  F1_mean
 Stunting  F0_Prenatal_Parto    0.6454   0.0102     0.5088     0.6891   0.4133
 Stunting      F1_Nacimiento    0.7374   0.0056     0.6406     0.7077   0.5054
 Stunting F2_Hospitalizacion    0.7405   0.0063     0.6391     0.7135   0.5080
 Stunting       F3_40semanas    0.7678   0.0081     0.6214     0.7583   0.5262
 Stunting          F4_3meses    0.8209   0.0094     0.6911     0.7793   0.5838
 Stunting     

In [6]:
# Grafica: evolucion del AUC a lo largo de la cascada
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fase_labels_short = {
    'F0_Prenatal_Parto':  'F0\nPrenatal',
    'F1_Nacimiento':      'F1\nNacimiento',
    'F2_Hospitalizacion': 'F2\nHosp.',
    'F3_40semanas':       'F3\n40 sem',
    'F4_3meses':          'F4\n3m',
    'F5_6meses':          'F5\n6m',
    'F6_9meses':          'F6\n9m',
}

for ax, (outcome_name, (_, color, _)) in zip(axes, OUTCOMES.items()):
    fases_ok = [f for f in FASE_ORDER if (outcome_name, f) in resultados]
    aucs     = [resultados[(outcome_name, f)]['AUC'].mean() for f in fases_ok]
    stds     = [resultados[(outcome_name, f)]['AUC'].std()  for f in fases_ok]
    x_lbls   = [fase_labels_short[f] for f in fases_ok]
    x        = np.arange(len(fases_ok))

    ax.plot(x, aucs, marker='o', linewidth=2.5, markersize=8, color=color)
    ax.fill_between(x,
                    np.array(aucs) - np.array(stds),
                    np.array(aucs) + np.array(stds),
                    alpha=0.15, color=color)
    for xi, auc_v, std_v in zip(x, aucs, stds):
        ax.text(xi, auc_v + 0.008, f'{auc_v:.3f}',
                ha='center', fontsize=9, fontweight='bold', color=color)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Aleatorio')
    ax.set_xticks(x)
    ax.set_xticklabels(x_lbls, fontsize=9)
    ax.set_ylim(0.45, 1.0)
    ax.set_ylabel('ROC-AUC')
    ax.set_title(f'{outcome_name}\n¿Cuando puede predecirse?', fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Evolucion del ROC-AUC por fase temporal — Cascada de prediccion',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mod_01_auc_cascada.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_01_auc_cascada.png')

Guardado: mod_01_auc_cascada.png


In [7]:
# Grafica: AUC, Sensibilidad, Especificidad para el mejor outcome (Stunting)
outcome_focus = 'Stunting'
color_focus   = '#e74c3c'

fases_ok = [f for f in FASE_ORDER if (outcome_focus, f) in resultados]
x_lbls   = [fase_labels_short[f] for f in fases_ok]
x        = np.arange(len(fases_ok))

met_rows = {
    'AUC':         [resultados[(outcome_focus,f)]['AUC'].mean()  for f in fases_ok],
    'Sensibilidad':[resultados[(outcome_focus,f)]['Sens'].mean() for f in fases_ok],
    'Especificidad':[resultados[(outcome_focus,f)]['Spec'].mean() for f in fases_ok],
    'F1':          [resultados[(outcome_focus,f)]['F1'].mean()   for f in fases_ok],
}

fig, ax = plt.subplots(figsize=(13, 5))
met_colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
for (met, vals), col in zip(met_rows.items(), met_colors):
    ax.plot(x, vals, marker='o', linewidth=2, markersize=7, color=col, label=met)

ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)
ax.set_xticks(x)
ax.set_xticklabels(x_lbls, fontsize=10)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('Metrica')
ax.set_title(f'Metricas por fase — {outcome_focus}\n'
             f'(AUC, Sensibilidad, Especificidad, F1)', fontweight='bold')
ax.legend(ncol=4)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT + 'mod_02_metricas_stunting.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_02_metricas_stunting.png')

Guardado: mod_02_metricas_stunting.png


In [8]:
# Curvas ROC por fase para Stunting (usando predicciones OOF)
fig, ax = plt.subplots(figsize=(8, 7))

palette = plt.cm.plasma(np.linspace(0.1, 0.9, len(fases_ok)))

for fase, col in zip(fases_ok, palette):
    oof, y_true = oof_preds[(outcome_focus, fase)]
    fpr, tpr, _ = roc_curve(y_true, oof)
    auc_v = roc_auc_score(y_true, oof)
    ax.plot(fpr, tpr, color=col, linewidth=2,
            label=f'{fase_labels_short[fase].replace(chr(10)," ")} (AUC={auc_v:.3f})')

ax.plot([0,1],[0,1],'k--', alpha=0.4, label='Aleatorio')
ax.set_xlabel('1 - Especificidad (FPR)')
ax.set_ylabel('Sensibilidad (TPR)')
ax.set_title(f'Curvas ROC por fase — {outcome_focus}\n(predicciones OOF)',
             fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT + 'mod_03_roc_curvas.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_03_roc_curvas.png')

Guardado: mod_03_roc_curvas.png


In [9]:
# Curvas Precision-Recall (mas informativa con desbalance de clases)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, (outcome_name, (_, color, _)) in zip(axes, OUTCOMES.items()):
    fases_out = [f for f in FASE_ORDER if (outcome_name, f) in oof_preds]
    palette_o = plt.cm.plasma(np.linspace(0.1, 0.9, len(fases_out)))

    for fase, col in zip(fases_out, palette_o):
        oof, y_true = oof_preds[(outcome_name, fase)]
        prec, rec, _ = precision_recall_curve(y_true, oof)
        ap = average_precision_score(y_true, oof)
        ax.plot(rec, prec, color=col, linewidth=1.8,
                label=f'{fase_labels_short[fase].replace(chr(10)," ")} (AP={ap:.3f})')

    baseline = y_true.mean()
    ax.axhline(baseline, color='gray', linestyle='--', alpha=0.5,
               label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Recall (Sensibilidad)')
    ax.set_ylabel('Precision')
    ax.set_title(f'{outcome_name}\nPrecision-Recall por fase', fontweight='bold')
    ax.legend(fontsize=7, loc='upper right')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)

plt.suptitle('Curvas Precision-Recall por fase y outcome (OOF)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mod_04_pr_curvas.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_04_pr_curvas.png')

Guardado: mod_04_pr_curvas.png


## 5. Interpretabilidad — SHAP Values

In [10]:
# Identificar la mejor fase por outcome para análisis SHAP
best_fase_per_outcome = {}
for outcome_name in OUTCOMES:
    best_auc  = -1
    best_fase = None
    for fase in FASE_ORDER:
        if (outcome_name, fase) in resultados:
            auc = resultados[(outcome_name, fase)]['AUC'].mean()
            if auc > best_auc:
                best_auc  = auc
                best_fase = fase
    best_fase_per_outcome[outcome_name] = (best_fase, best_auc)
    print(f'{outcome_name}: mejor fase = {best_fase} (AUC={best_auc:.4f})')

# Tambien calcular SHAP para F1 (solo nacimiento) — valor clinico maximo
print('\nFases elegidas para SHAP:')
print('  - Mejor fase por AUC (por outcome)')
print('  - F1_Nacimiento (para comparar con solo datos de nacimiento)')

Stunting: mejor fase = F6_9meses (AUC=0.9290)
Bajo_peso: mejor fase = F6_9meses (AUC=0.9634)
Wasting: mejor fase = F6_9meses (AUC=0.9245)

Fases elegidas para SHAP:
  - Mejor fase por AUC (por outcome)
  - F1_Nacimiento (para comparar con solo datos de nacimiento)


In [11]:
# SHAP para Stunting — mejor fase
outcome_shap = 'Stunting'
best_fase_shap, _ = best_fase_per_outcome[outcome_shap]
X_shap, y_shap   = data_store[(outcome_shap, best_fase_shap)]

# Usar el primer modelo del fold (entrenado en ~80% de datos)
model_shap = best_models[(outcome_shap, best_fase_shap)][0]

print(f'Calculando SHAP para {outcome_shap} — {best_fase_shap}')
print(f'  Dataset: {X_shap.shape[0]:,} muestras x {X_shap.shape[1]} features')

# Muestra representativa para SHAP (max 3000 filas por velocidad)
n_shap = min(3000, len(X_shap))
X_sample = X_shap.sample(n_shap, random_state=42)

explainer   = shap.TreeExplainer(model_shap)
shap_values = explainer.shap_values(X_sample)

print(f'  SHAP values calculados: shape={np.array(shap_values).shape}')

Calculando SHAP para Stunting — F6_9meses
  Dataset: 30,953 muestras x 198 features


  SHAP values calculados: shape=(3000, 198)


In [12]:
# SHAP Summary Plot — Top 20 factores de riesgo
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values, X_sample,
    max_display=20,
    plot_type='dot',
    show=False
)
plt.title(f'SHAP — Top 20 factores de riesgo\n'
          f'{outcome_shap} | {best_fase_shap}\n'
          f'(rojo=aumenta riesgo, azul=disminuye riesgo)',
          fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig(OUT + 'mod_05_shap_summary.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_05_shap_summary.png')

Guardado: mod_05_shap_summary.png


In [13]:
# SHAP Bar plot — importancia global de features
shap_importance = pd.DataFrame({
    'feature':    X_sample.columns,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(25)

fig, ax = plt.subplots(figsize=(10, 8))
colors_shap = ['#e74c3c' if i < 5 else '#e67e22' if i < 10 else '#3498db'
               for i in range(len(shap_importance))]
bars = ax.barh(shap_importance['feature'][::-1],
               shap_importance['mean_abs_shap'][::-1],
               color=colors_shap[::-1], edgecolor='white')
ax.set_xlabel('Mean |SHAP value| (impacto promedio en la prediccion)')
ax.set_title(f'Importancia global de features — SHAP\n'
             f'{outcome_shap} | {best_fase_shap}',
             fontweight='bold')

# Etiquetas de valor
for bar, v in zip(bars, shap_importance['mean_abs_shap'][::-1]):
    ax.text(v + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT + 'mod_06_shap_importancia.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_06_shap_importancia.png')
print()
print('Top 10 factores de riesgo (SHAP):')
for _, row in shap_importance.head(10).iterrows():
    print(f'  {row["feature"]:<40} SHAP={row["mean_abs_shap"]:.4f}')

Guardado: mod_06_shap_importancia.png

Top 10 factores de riesgo (SHAP):
  zscoretalla9                             SHAP=1.3241
  zscoretalla6                             SHAP=0.6952
  zscorepeso9                              SHAP=0.1527
  velocidad9_6mesesOMS                     SHAP=0.1122
  zscoretalla9cat                          SHAP=0.0928
  zscorepeso6                              SHAP=0.0580
  zscoretalla2                             SHAP=0.0555
  zscoretalla6cat                          SHAP=0.0552
  velocidad6_3mesesOMS                     SHAP=0.0335
  CP_TallaMadre                            SHAP=0.0286


## 6. Sistema de Riesgo Dinámico
Visualización de cómo evoluciona la probabilidad de riesgo de un paciente a través de las fases.

In [14]:
# Calcular probabilidades en cada fase para todos los pacientes
# Solo para Stunting (outcome principal)
# Para cada paciente que tenga datos en TODAS las fases, calculamos la trayectoria

outcome_dyn = 'Stunting'
target_dyn  = OUTCOMES[outcome_dyn][0]

# Pacientes con outcome conocido
df_known = df[df[target_dyn].notna()].copy()
print(f'Pacientes con outcome {outcome_dyn}: {len(df_known):,}')

# Para cada fase, predecir probabilidad usando el primer modelo del fold
prob_cols = {}
for fase in FASE_ORDER:
    if (outcome_dyn, fase) not in best_models:
        continue
    model_f = best_models[(outcome_dyn, fase)][0]
    feats_f = cumulative_features[fase]
    cols_f  = [c for c in feats_f if c in df_known.columns]

    X_pred = df_known[cols_f].copy()
    probs  = model_f.predict(X_pred)
    col_name = f'prob_{fase}'
    df_known[col_name] = probs
    prob_cols[fase] = col_name

print(f'Probabilidades calculadas para {len(prob_cols)} fases')
print(f'Columnas: {list(prob_cols.values())}')

Pacientes con outcome Stunting: 30,953


Probabilidades calculadas para 7 fases
Columnas: ['prob_F0_Prenatal_Parto', 'prob_F1_Nacimiento', 'prob_F2_Hospitalizacion', 'prob_F3_40semanas', 'prob_F4_3meses', 'prob_F5_6meses', 'prob_F6_9meses']


In [15]:
# Visualizar trayectorias de riesgo individual
# Seleccionar muestras representativas: alto riesgo real, bajo riesgo real

fases_prob = list(prob_cols.keys())
col_probs  = list(prob_cols.values())

# Pacientes con todas las probabilidades calculadas
df_traj = df_known[col_probs + [target_dyn]].dropna()

# Tomar muestras: 10 pacientes stunted y 10 no stunted
stunted     = df_traj[df_traj[target_dyn] == 1].sample(min(15, (df_traj[target_dyn]==1).sum()),
                                                         random_state=42)
no_stunted  = df_traj[df_traj[target_dyn] == 0].sample(min(15, (df_traj[target_dyn]==0).sum()),
                                                         random_state=42)

x_fases = [fase_labels_short[f].replace('\n',' ') for f in fases_prob]
x       = np.arange(len(fases_prob))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Panel izquierdo: trayectorias individuales
ax = axes[0]
for _, row in no_stunted.iterrows():
    vals = [row[c] for c in col_probs]
    ax.plot(x, vals, color='#3498db', alpha=0.3, linewidth=1)
for _, row in stunted.iterrows():
    vals = [row[c] for c in col_probs]
    ax.plot(x, vals, color='#e74c3c', alpha=0.5, linewidth=1.5)

# Lineas medias
ax.plot(x, [no_stunted[c].mean() for c in col_probs],
        color='#2980b9', linewidth=3, label='Media NO stunted', zorder=5)
ax.plot(x, [stunted[c].mean() for c in col_probs],
        color='#c0392b', linewidth=3, label='Media STUNTED', zorder=5)

ax.axhline(0.5, color='black', linestyle='--', alpha=0.4, label='Umbral 0.5')
ax.set_xticks(x)
ax.set_xticklabels(x_fases, fontsize=9)
ax.set_ylabel('P(Stunting a 12m)')
ax.set_title('Trayectorias de riesgo individual\n(rojo=stunted, azul=normal)', fontweight='bold')
ax.legend(fontsize=8)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# Panel derecho: distribucion de probabilidades por fase
ax2 = axes[1]
offsets = np.linspace(-0.25, 0.25, len(fases_prob))
for i, (fase, col) in enumerate(zip(fases_prob, col_probs)):
    p_stunted    = stunted[col].values
    p_no_stunted = no_stunted[col].values
    pos = i
    bp1 = ax2.boxplot(p_stunted, positions=[pos - 0.2], widths=0.35,
                      patch_artist=True,
                      boxprops=dict(facecolor='#e74c3c', alpha=0.6),
                      medianprops=dict(color='darkred', linewidth=2),
                      whiskerprops=dict(color='#e74c3c'),
                      capprops=dict(color='#e74c3c'),
                      flierprops=dict(marker='.', markersize=3, color='#e74c3c'))
    bp2 = ax2.boxplot(p_no_stunted, positions=[pos + 0.2], widths=0.35,
                      patch_artist=True,
                      boxprops=dict(facecolor='#3498db', alpha=0.6),
                      medianprops=dict(color='darkblue', linewidth=2),
                      whiskerprops=dict(color='#3498db'),
                      capprops=dict(color='#3498db'),
                      flierprops=dict(marker='.', markersize=3, color='#3498db'))

ax2.axhline(0.5, color='black', linestyle='--', alpha=0.4)
ax2.set_xticks(range(len(fases_prob)))
ax2.set_xticklabels(x_fases, fontsize=9)
ax2.set_title('Distribucion de riesgo por fase\n(rojo=stunted, azul=normal)', fontweight='bold')
ax2.set_ylabel('P(Stunting a 12m)')

red_patch  = mpatches.Patch(color='#e74c3c', alpha=0.6, label='Stunted (real)')
blue_patch = mpatches.Patch(color='#3498db', alpha=0.6, label='Normal (real)')
ax2.legend(handles=[red_patch, blue_patch], fontsize=9)
ax2.grid(True, alpha=0.3)

plt.suptitle('Sistema de Riesgo Dinamico — Stunting a 12 meses EC',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT + 'mod_07_riesgo_dinamico.png', bbox_inches='tight')
plt.close()
print('Guardado: mod_07_riesgo_dinamico.png')
print()
# Separacion media entre grupos en cada fase
print('Separacion media entre grupos por fase:')
for fase, col in zip(fases_prob, col_probs):
    med_s  = stunted[col].mean()
    med_ns = no_stunted[col].mean()
    print(f'  {fase_labels_short[fase].replace(chr(10)," "):<20}: stunted={med_s:.3f}  normal={med_ns:.3f}  delta={med_s-med_ns:.3f}')

Guardado: mod_07_riesgo_dinamico.png

Separacion media entre grupos por fase:
  F0 Prenatal         : stunted=0.562  normal=0.457  delta=0.106
  F1 Nacimiento       : stunted=0.628  normal=0.421  delta=0.207
  F2 Hosp.            : stunted=0.642  normal=0.382  delta=0.260
  F3 40 sem           : stunted=0.703  normal=0.338  delta=0.365
  F4 3m               : stunted=0.742  normal=0.289  delta=0.454
  F5 6m               : stunted=0.767  normal=0.183  delta=0.584
  F6 9m               : stunted=0.787  normal=0.145  delta=0.641


## 7. Baseline: Regresión Logística L1 (comparación)

In [16]:
# Baseline con Logistic Regression + L1 para las 156 variables universales
# Usar solo el outcome principal (Stunting) y la mejor fase

outcome_bl = 'Stunting'
fase_bl    = best_fase_per_outcome[outcome_bl][0]
X_bl, y_bl = data_store[(outcome_bl, fase_bl)]

# Imputar medianas (LR requiere datos completos)
X_bl_imp = X_bl.fillna(X_bl.median())

# Pipeline: escalado + LR L1
pipe_l1 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LogisticRegression(
                    penalty='l1', solver='liblinear',
                    C=0.1, class_weight='balanced',
                    max_iter=1000, random_state=42))
])

skf_bl = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
aucs_bl = []
for tr, val in skf_bl.split(X_bl_imp, y_bl):
    pipe_l1.fit(X_bl_imp.iloc[tr], y_bl.iloc[tr])
    prob_val = pipe_l1.predict_proba(X_bl_imp.iloc[val])[:,1]
    aucs_bl.append(roc_auc_score(y_bl.iloc[val], prob_val))

auc_lgbm_best = best_fase_per_outcome[outcome_bl][1]
print(f'Baseline — Logistic Regression L1')
print(f'  AUC media (5-fold): {np.mean(aucs_bl):.4f} ± {np.std(aucs_bl):.4f}')
print()
print(f'LightGBM ({fase_bl})')
print(f'  AUC media (5-fold): {auc_lgbm_best:.4f}')
print()
print(f'Ganancia LightGBM vs Logistica: +{auc_lgbm_best - np.mean(aucs_bl):.4f}')

# Coeficientes no-cero de la regresión logística (variables seleccionadas por L1)
pipe_l1.fit(X_bl_imp.fillna(X_bl_imp.median()), y_bl)
coef = pd.Series(pipe_l1.named_steps['lr'].coef_[0], index=X_bl_imp.columns)
coef_nz = coef[coef != 0].sort_values(key=abs, ascending=False)
print(f'\nVariables seleccionadas por L1: {len(coef_nz)} de {len(coef)}')
print(f'Top 15 coeficientes (L1):')
for feat, val in coef_nz.head(15).items():
    signo = '+' if val > 0 else ''
    print(f'  {feat:<40} coef = {signo}{val:.4f}')

Baseline — Logistic Regression L1
  AUC media (5-fold): 0.9216 ± 0.0031

LightGBM (F6_9meses)
  AUC media (5-fold): 0.9290

Ganancia LightGBM vs Logistica: +0.0074



Variables seleccionadas por L1: 153 de 198
Top 15 coeficientes (L1):
  zscoretalla9                             coef = -1.3044
  zscoretalla6                             coef = -0.9151
  velocidad9_6mesesOMS                     coef = -0.3100
  zscorepeso6                              coef = -0.2928
  zscoretalla2                             coef = -0.2486
  zscorepeso9                              coef = -0.2273
  zscorepesotalla2                         coef = +0.1984
  vino9m                                   coef = -0.1740
  zscoretalla9cat                          coef = -0.1666
  vino6m                                   coef = -0.1625
  zscoretalla6cat                          coef = -0.1498
  zscorepesotalla9                         coef = +0.1353
  zscorepesotalla9cat                      coef = -0.1216
  ali9m                                    coef = -0.1157
  BMI2                                     coef = -0.1060


## 8. Resumen Final del Pipeline

In [17]:
print('=' * 65)
print('RESUMEN PIPELINE DE MODELADO — Malnutricion PMCI')
print('=' * 65)

print('\nMEJOR AUC POR OUTCOME:')
for outcome, (fase, auc) in best_fase_per_outcome.items():
    _, _, spw = OUTCOMES[outcome]
    print(f'  {outcome:<12}: {auc:.4f} AUC (fase={fase})')

print('\nCASCADA TEMPORAL — AUC STUNTING:')
for fase in FASE_ORDER:
    if ('Stunting', fase) in resultados:
        mdf  = resultados[('Stunting', fase)]
        feats = len(cumulative_features[fase])
        print(f'  {fase_labels_short[fase].replace(chr(10)," "):<22}'
              f': AUC={mdf["AUC"].mean():.4f}  '
              f'Sens={mdf["Sens"].mean():.3f}  '
              f'Spec={mdf["Spec"].mean():.3f}  '
              f'({feats} features)')

print(f'''\nCONCLUSIONES:
  1. LightGBM supera a Logistica L1 en AUC
  2. La senal predictiva aumenta con cada fase (validando la cascada temporal)
  3. Ya desde F1 (nacimiento) hay capacidad predictiva util
  4. SHAP identifica los factores de riesgo mas relevantes por fase
  5. El sistema dinamico muestra separacion entre grupos desde etapas tempranas

ARCHIVOS GENERADOS:
  mod_01_auc_cascada.png       — AUC por fase y outcome
  mod_02_metricas_stunting.png — Metricas detalladas Stunting
  mod_03_roc_curvas.png        — Curvas ROC por fase
  mod_04_pr_curvas.png         — Curvas Precision-Recall
  mod_05_shap_summary.png      — SHAP top 20 factores
  mod_06_shap_importancia.png  — Importancia global SHAP
  mod_07_riesgo_dinamico.png   — Trayectorias de riesgo
''')

RESUMEN PIPELINE DE MODELADO — Malnutricion PMCI

MEJOR AUC POR OUTCOME:
  Stunting    : 0.9290 AUC (fase=F6_9meses)
  Bajo_peso   : 0.9634 AUC (fase=F6_9meses)
  Wasting     : 0.9245 AUC (fase=F6_9meses)

CASCADA TEMPORAL — AUC STUNTING:
  F0 Prenatal           : AUC=0.6454  Sens=0.509  Spec=0.689  (41 features)
  F1 Nacimiento         : AUC=0.7374  Sens=0.641  Spec=0.708  (75 features)
  F2 Hosp.              : AUC=0.7405  Sens=0.639  Spec=0.713  (107 features)
  F3 40 sem             : AUC=0.7678  Sens=0.621  Spec=0.758  (137 features)
  F4 3m                 : AUC=0.8209  Sens=0.691  Spec=0.779  (164 features)
  F5 6m                 : AUC=0.8935  Sens=0.792  Spec=0.823  (183 features)
  F6 9m                 : AUC=0.9290  Sens=0.834  Spec=0.859  (198 features)

CONCLUSIONES:
  1. LightGBM supera a Logistica L1 en AUC
  2. La senal predictiva aumenta con cada fase (validando la cascada temporal)
  3. Ya desde F1 (nacimiento) hay capacidad predictiva util
  4. SHAP identifica los fa